In [26]:
pip install pandas numpy scikit-learn sentence-transformers tensorflow openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd

df = pd.read_csv("games_merged.csv.xls")

print(df.shape)
print(df.columns)

(3500, 14)
Index(['id', 'name', 'slug', 'summary', 'rating', 'rating_count',
       'total_rating', 'total_rating_count', 'release_date', 'cover_url',
       'genres', 'platforms', 'companies', 'websites'],
      dtype='object')


ver os nomes das colunas

In [29]:
df = df[
    [
        "name",
        "summary",
        "genres",
        "platforms",
        "rating"
    ]
].copy()

df.fillna("", inplace=True)

processar generos

In [30]:
from sklearn.preprocessing import MultiLabelBinarizer

df["genres"] = df["genres"].apply(
    lambda x: [i.strip() for i in str(x).split(",")]
)

mlb_genres = MultiLabelBinarizer()

genres_matrix = mlb_genres.fit_transform(
    df["genres"]
)

processar plataformas

In [31]:
df["platforms"] = df["platforms"].apply(
    lambda x: [i.strip() for i in str(x).split(",")]
)

mlb_platforms = MultiLabelBinarizer()

platform_matrix = mlb_platforms.fit_transform(
    df["platforms"]
)

embeddings dos summaries

In [32]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

summary_embeddings = model.encode(
    df["summary"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 110/110 [00:34<00:00,  3.21it/s]


rating 

In [33]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

ratings = scaler.fit_transform(
    df[["rating"]].fillna(0)
)

In [34]:
import numpy as np

X = np.concatenate(
    [
        summary_embeddings,
        genres_matrix,
        platform_matrix,
        ratings
    ],
    axis=1
)

print(X.shape)

(3500, 711)


In [35]:
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model

input_dim = X.shape[1]

inputs = Input(shape=(input_dim,))

x = Dense(
    256,
    activation="relu"
)(inputs)

x = Dense(
    128,
    activation="relu"
)(x)

latent = Dense(
    64,
    activation="relu",
    name="embedding"
)(x)

x = Dense(
    128,
    activation="relu"
)(latent)

x = Dense(
    256,
    activation="relu"
)(x)

outputs = Dense(
    input_dim,
    activation="linear"
)(x)

autoencoder = Model(
    inputs,
    outputs
)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 711)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │       182,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 711)            │       182,727 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 447,495 (1.71 MB)

 Trainable params: 447,495 (1.71 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
history = autoencoder.fit(
    X,
    X,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0082 - val_loss: 0.0071
Epoch 2/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0058 - val_loss: 0.0056
Epoch 3/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0047 - val_loss: 0.0049
Epoch 4/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0041 - val_loss: 0.0045
Epoch 5/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0038 - val_loss: 0.0042
Epoch 6/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0035 - val_loss: 0.0040
Epoch 7/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0033 - val_loss: 0.0038
Epoch 8/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0031 - val_loss: 0.0037
Epoch 9/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0030 - val_loss: 0.0035
Epoch 10/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0029 - val_loss: 0.0035
Epoch 11/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0028 - val_loss: 0.0034
Epoch 12/200
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0

In [37]:
encoder = Model(
    autoencoder.input,
    autoencoder.get_layer(
        "embedding"
    ).output
)

game_embeddings = encoder.predict(X)

print(game_embeddings.shape)

# (3500,64)

110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
(3500, 64)


In [38]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    game_embeddings
)

In [39]:
def recommend_game(
    game_name,
    top_k=10
):

    idx = df[
        df["name"].str.lower()
        ==
        game_name.lower()
    ].index

    if len(idx) == 0:
        return None

    idx = idx[0]

    scores = similarity_matrix[idx]

    top_indices = (
        scores.argsort()[::-1]
    )[1:top_k+1]

    return df.iloc[
        top_indices
    ][
        [
            "name",
            "rating"
        ]
    ]

In [40]:
recommend_game(
    "Grand Theft Auto V"
)

,name,rating
87,Wolfenstein: The New Order,81.624119
39,Grand Theft Auto IV,83.422800
702,Vanquish,79.413286
207,Saints Row: The Third,78.145947
2074,Transformers: Fall of Cybertron,73.118556
4,Grand Theft Auto: San Andreas,90.107753
96,Mafia II,81.551681
275,Saints Row IV,74.358235
350,Just Cause 2,77.093766
1776,Grand Theft Auto V: Special Edition,99.310452


In [41]:
def recommend_for_user(
    games_played,
    top_k=10
):

    played_indices = []

    for game in games_played:

        idx = df[
            df["name"] == game
        ].index

        if len(idx):
            played_indices.append(idx[0])

    user_vector = np.mean(
        game_embeddings[played_indices],
        axis=0
    )

    scores = cosine_similarity(
        user_vector.reshape(1, -1),
        game_embeddings
    )[0]

    scores[played_indices] = -1

    top_indices = scores.argsort()[::-1][:top_k]

    return df.iloc[top_indices][
        ["name", "rating"]
    ]

In [42]:
recommend_for_user(
    [
        "Grand Theft Auto V",
        "Red Dead Redemption 2",
        "Cyberpunk 2077"
    ]
)

,name,rating
205,Watch Dogs 2,76.736106
1566,Outriders,74.139051
282,Borderlands 3,76.330924
545,Tom Clancy's The Division 2,75.001794
242,Destiny 2,74.414875
525,Hitman 2,81.054035
83,Rise of the Tomb Raider,81.380212
565,Tom Clancy's Ghost Recon: Wildlands,72.550851
284,Resident Evil Village,83.807648
158,Control,83.135655


In [43]:
autoencoder.save(
    "game_recommender.keras"
)

In [44]:
import pickle

with open(
    "game_embeddings.pkl",
    "wb"
) as f:

    pickle.dump(
        game_embeddings,
        f
    )